# BENCHMARK THUC NGHIEM CEGR: DOORKEY-5x5 (MUC 1 - MOI TRUONG CHUAN)

**Thực nghiệm chuẩn hóa CEGR:** So sánh toàn diện 4 phương pháp chính (`EUREKA`, `CARD`, `CARD_REFL`, `CEGR`) cùng 6 biến thể Ablation (`CEGR_noPool`, `CEGR_noAnchors`, `CEGR_noClass`, `CEGR_noG2`, `CEGR_noG4`, `CEGR_noG5`) trên môi trường **doorkey5**.

- **Tối ưu hóa mô-đun:** Sử dụng trực tiếp gói thư viện chuẩn `cegr` ([cegr/](file:///d:/CEGR/Cegr-official/cegr)) thay vì định nghĩa lại mã nguồn thủ công.
- **Trọn vẹn quy trình 3 pha:** Pha 1 (Sinh dataset mutants + natural hack), Pha 2 (Vòng lặp sửa đổi + GateKeeper 5 cổng + LLM feedback), Pha 3 (Bootstrap thống kê, biểu đồ và 4 bảng LaTeX).


In [ ]:
# =====================================================================
# CELL 0: CLONE KHO MÃ NGUỒN CEGR (KAGGLE / GOOGLE COLAB SETUP)
# =====================================================================
import os
import sys

# 1. Tự động clone kho mã nguồn CEGR nếu chạy trên Colab / Kaggle và chưa có mã nguồn
if not os.path.exists("cegr") and not os.path.exists("../cegr"):
    if not os.path.exists("CEGR"):
        print(">> Đang clone kho mã nguồn CEGR từ GitHub...")
        !git clone https://github.com/ToanThinh1208/CEGR.git
    else:
        print(">> Thư mục CEGR đã tồn tại trên đĩa.")
        # Bỏ comment nếu muốn tự động kéo code mới nhất từ GitHub:
        # !git -C CEGR pull
    
    if os.path.exists("CEGR"):
        %cd CEGR
elif os.path.exists("CEGR") and not os.path.exists("cegr"):
    %cd CEGR

# 2. Đảm bảo thư mục gốc repo nằm trong sys.path để import cegr
repo_root = os.path.abspath(".") if os.path.exists("cegr") else (os.path.abspath("..") if os.path.exists("../cegr") else os.path.abspath("CEGR"))
for p in [repo_root, os.path.abspath("."), os.path.abspath("..")]:
    if os.path.exists(os.path.join(p, "cegr")) and p not in sys.path:
        sys.path.insert(0, p)

# 3. Thông báo kiểm tra môi trường
print(f"✅ Thư mục làm việc hiện tại: {os.getcwd()}")
print(f"✅ Repo root trong sys.path: {repo_root}")

try:
    import cegr
    print(f"✅ Đã sẵn sàng nạp package 'cegr' tại: {cegr.__file__}")
except ImportError as e:
    print(f"⚠️ Lưu ý: Chưa thể nạp package cegr ({e}). Hãy đảm bảo bạn đang ở thư mục repo CEGR.")


In [ ]:
# =====================================================================
# CELL 1: CÀI ĐẶT THƯ VIỆN & CẤU HÌNH TOÀN CỤC (+ cấu hình CARD-CEGR)
# =====================================================================
import subprocess, sys, os

# Gỡ các gói gây xung đột dependency trên Kaggle trước khi cài Torch + vLLM
!pip uninstall -y torchvision torchaudio cudf-cu12 cuml-cu12 dask-cuda pylibcudf-cu12 > /dev/null 2>&1
!pip install -q torch==2.6.0+cu124 --index-url https://download.pytorch.org/whl/cu124
!pip install -q vllm transformers accelerate tiktoken autoawq hf_transfer

# Đảm bảo đường dẫn import cegr package hoạt động cả khi chạy từ ./notebooks hoặc root
for p in [os.path.abspath('..'), os.path.abspath('.')]:
    if p not in sys.path: sys.path.insert(0, p)

import json, time, random, collections, hashlib, math, re, ast, builtins, itertools
import signal, threading
from dataclasses import dataclass, field, asdict
from typing import Dict, List, NamedTuple, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mp

# Lá cờ chọn môi trường thực thi: True = Colab, False = Kaggle
IS_COLAB = False   # True: Colab | False: Kaggle (Người dùng tự điền True hoặc False)


# # 2. Dùng biến tăng tốc mới của HuggingFace (thay cho HF_HUB_ENABLE_HF_TRANSFER đã bị deprecated)
# os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# LƯU Ý: Không hardcode HF_TOKEN ở đây. Qwen2.5-Coder-7B-Instruct-AWQ là model mở hoàn toàn.

plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 300,
    "axes.grid": True, "grid.alpha": 0.25,
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10
})

# ============== CẤU HÌNH THỰC NGHIỆM CHUẨN HÓA =============
MODEL_ID         = "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"
MAX_MODEL_LEN    = 6144

FAST_SMOKE_TEST   = False
# Cấu hình Sinh tập Dataset (Pha A)
N_NATURAL_PROMPTS = 30  if FAST_SMOKE_TEST else 360 # Sinh 360 mẫu bằng Batch vLLM (~40 giây) để lọc lấy >= 18-25 mẫu natural hacked
TARGET_MAIN_N     = 8   if FAST_SMOKE_TEST else 16 # Mục tiêu tối thiểu trong tập Main (Mutant + Natural)
PILOT_N           = 2   if FAST_SMOKE_TEST else 4   # Số mẫu tách riêng làm Pilot kiểm tra cổng G2
    

# ============== [CEGR] CẤU HÌNH THỰC NGHIỆM (thay CONDS/K_WAVES cũ) ==============
ENV_NAME         = "doorkey5"   # "doorkey5"
N_ROUNDS         = 4            # ngân sách gọi Coder / job (CÙNG ngân sách cho mọi phương pháp; EUREKA = 2 candidate x 2 vòng)
N_SAMPLES        = 1 if FAST_SMOKE_TEST else 3   # seed LLM độc lập cho mỗi (reward gốc, phương pháp)
GEN_TEMP         = 0.95
REPAIR_TEMP      = 0.35
REPAIR_MAXTOK    = 900
EUREKA_POP       = 2            # số candidate / vòng của baseline Eureka-style

METHODS          = ["EUREKA", "CARD", "CARD_REFL", "CEGR"]
ABLATIONS        = ["CEGR_noPool", "CEGR_noAnchors", "CEGR_noClass", "CEGR_noG2", "CEGR_noG4", "CEGR_noG5"]
CONDS            = METHODS + ABLATIONS
N_MAIN_EVAL      = 4  if FAST_SMOKE_TEST else 244   # số reward gốc (planted) chạy đủ 4 phương pháp
N_ABL_EVAL       = 2  if FAST_SMOKE_TEST else 12   # số reward gốc chạy thêm 6 ablation
N_SCRATCH_STARTS = 1  if FAST_SMOKE_TEST else 6   # số điểm xuất phát "sinh từ đầu" (hack tự nhiên)

# --- GateKeeper / CE ---
EPS_SIGNAL_REL   = 0.02         # eps tương đối so với |return quỹ đạo thành công tham chiếu|
EPS_SIGNAL_ABS   = 1e-3
RHO_THR          = 0.75         # ngưỡng Spearman(reward, tiến độ thật) ở Gate 4
M_CE_PER_ROUND   = 2            # tối đa CE thêm vào pool từ mỗi nguồn / vòng
POOL_MAX         = 40
STALL_K          = 2            # dừng khi pool không thêm phần tử mới sau k vòng liên tiếp (chỉ tính khi rớt cổng >= 3)
MAX_INVARIANTS   = 8
FEEDBACK_MAX_CHARS = 4500
ATTACKER_MODE    = "dp"         # "dp": attacker = DP chính xác (env nhỏ) | "none": chỉ dựa TPE + RL probe
ARCHITECT_USE_LLM = True
N_P0_NOISY, MAX_P0_NOISY_PAIRS = 24, 20
N_SIGNAL_TRAJ, N_EVAL_TRAJ = 40, 60
CE_STREAM_SEED, SIGNAL_STREAM_SEED, EVAL_STREAM_SEED = 1000, 2000, 3000   # 3 luồng seed TÁCH BIỆT
SHORT_PROBE_EPISODES, SHORT_PROBE_SEEDS = 1000, 2     # Gate 5 (rẻ); seed gate 5 = SEED+100+s
FULL_RL_EPISODES = 800 if FAST_SMOKE_TEST else 2500   # RL đầy đủ cho R* cuối
FINAL_RL_SEEDS, FINAL_RL_SEED_BASE = 5, 9000          # seed đánh giá cuối, tách khỏi seed gate 5

# Cấu hình Chấm điểm & Thống kê (Pha C)
# ============== 1. CÔNG TẮC CHẾ ĐỘ CHẠY (CHỌN BẢN MỚI HOẶC CŨ) ==============
# Đặt thẳng sang "relaxed" và "agent_pos" vì bạn đang chạy thực nghiệm bản mới.
# (Khi nào cần kiểm tra hồi quy bản cũ thì chỉ cần đổi lại thành "strict_v1" và "none")
SCORING_PROFILE  = "relaxed"     # Chọn: "relaxed" (bản mới) hoặc "strict_v1" (bản cũ)
KEYPOS_MODE      = "agent_pos"   # Chọn: "agent_pos" (bản mới) hoặc "none" (bản cũ)
G2_MODE = "merge"   # "hard" (cũ) | "merge" | "soft"
PROFILES = {
    "strict_v1": dict(
        probe_episodes=1200, tail_thr=0.66, probe_gamma=0.99,
        optimistic_init=0.0, alpha_decay=False, require_dense=True
    ),
    "relaxed": dict(
        probe_episodes=2500, tail_thr=0.50, probe_gamma=1.0,
        optimistic_init=1.0, alpha_decay=True, require_dense=True
    ),
}
assert SCORING_PROFILE in PROFILES, f"Profile không hợp lệ: {SCORING_PROFILE}"
assert KEYPOS_MODE in ("none", "agent_pos"), f"KEYPOS_MODE không hợp lệ: {KEYPOS_MODE}"

ACTIVE_PROFILE   = PROFILES[SCORING_PROFILE]
SUFFIX           = f"__{ENV_NAME}__{SCORING_PROFILE}__{KEYPOS_MODE}"

# ============== 2. CẤU HÌNH CHẤM ĐIỂM & THỐNG KÊ (ĐỒNG BỘ TỰ ĐỘNG) ==============
HORIZONS         = (30, 60)                        # DP kép ở T=30 và T=60
EPS_MARGIN_REL   = 0.01                            # Ngưỡng margin tương đối M(R') > 1% |V_succ|
PROBE_EPISODES   = ACTIVE_PROFILE["probe_episodes"] # Tự động lấy 2500 (relaxed) hoặc 1200 (strict_v1)
PROBE_GAMMA      = ACTIVE_PROFILE["probe_gamma"]    # Tự động lấy 1.0 (relaxed) hoặc 0.99 (strict_v1)
PROBE_TAIL_THR   = ACTIVE_PROFILE["tail_thr"]       # Tự động lấy 0.50 (relaxed) hoặc 0.66 (strict_v1)
OPTIMISTIC_INIT  = ACTIVE_PROFILE["optimistic_init"]# Tự động lấy 1.0 (relaxed) hoặc 0.0 (strict_v1)
ALPHA_DECAY      = ACTIVE_PROFILE["alpha_decay"]    # Tự động lấy True (relaxed) hoặc False (strict_v1)
REQUIRE_DENSE    = ACTIVE_PROFILE["require_dense"]
PROBE_SEEDS      = 3                               # 3 seeds Q-learning cho mỗi bản sửa qua cổng DP
N_BOOT           = 5000                            # Số lần lấy mẫu Cluster Bootstrap cấp độ reward
SEED             = 42

# ============== 3. QUẢN LÝ ĐƯỜNG DẪN & HẬU TỐ FILE OUTPUT ==============
# Điều kiện môi trường: True = Colab, False = Kaggle (dựa vào biến IS_COLAB)
ON_COLAB  = IS_COLAB
ON_KAGGLE = not IS_COLAB

if IS_COLAB:
    OUT_DIR    = "/content/cegr_out"
    LEGACY_DIR = "/content/cegr_out"
else:
    OUT_DIR    = "/kaggle/working/cegr"
    LEGACY_DIR = "/kaggle/input/cegr-v1-results" if os.path.exists("/kaggle/input/cegr-v1-results") else OUT_DIR

os.makedirs(OUT_DIR, exist_ok=True)

def out_path(base_name: str, ext: str, use_suffix: bool = True) -> str:
    """Sinh đường dẫn lưu file trong OUT_DIR có gắn hậu tố __SCORING_PROFILE__KEYPOS_MODE."""
    sfx = SUFFIX if use_suffix else ""
    return os.path.join(OUT_DIR, f"{base_name}{sfx}{ext}")

def find_legacy_file(filename: str) -> str:
    """Tìm file gốc (manifest.json / results_main.jsonl) ở OUT_DIR hoặc LEGACY_DIR nếu có."""
    p_out = os.path.join(OUT_DIR, filename)
    if os.path.exists(p_out):
        return p_out
    p_leg = os.path.join(LEGACY_DIR, filename)
    if os.path.exists(p_leg):
        return p_leg
    return p_out

random.seed(SEED); np.random.seed(SEED)
print(f"✓ Cấu hình hoàn tất | Mode = {SUFFIX}")
print(f"  -> Probe: episodes={PROBE_EPISODES}, gamma={PROBE_GAMMA}, tail_thr={PROBE_TAIL_THR}, opt_init={OPTIMISTIC_INIT}, alpha_decay={ALPHA_DECAY}")
print(f"  -> OUT_DIR = {OUT_DIR}")


# CELL 2: MÔI TRƯỜNG DOORKEY, SANDBOX AN TOÀN & BỘ GIẢI DP CHÍNH XÁC (TỪ CEGR)


In [ ]:
# =====================================================================
# CELL 2: KHỞI TẠO MÔI TRƯỜNG, SANDBOX & BỘ GIẢI DP TỪ CEGR
# =====================================================================
from cegr.envs.doorkey import DoorKeyEnv, make_env, ENV_CONFIGS, State, ACTION_NAMES
from cegr.core.code_sandbox import (
    RewardError, check_code, code_fingerprint, ast_node_count,
    make_namespace, build_reward_table, RewardTable
)
from cegr.solvers.dp_solver import solve, analyze, Analysis, Step, Trajectory

env = make_env(ENV_NAME)
print(f"✓ Môi trường {ENV_NAME} sẵn sàng: {env.n} states khả đạt, horizon={env.horizon} (T1={env.horizon}, T2={2*env.horizon}).")


# CELL 3: ORACLE (ABLATION CULPRIT, SUB-GOAL SHAPING, VERIFIED FIX) & Q-LEARNING PROBE (TỪ CEGR)


In [ ]:
# =====================================================================
# CELL 3: ORACLE & Q-LEARNING PROBE TỪ CEGR
# =====================================================================
from cegr.solvers.rl_probe import evaluate_qlearning_probe_v2, run_probe
from cegr.solvers.verified_fix import (
    evaluate_verified_fix, culprit_ablation_oracle, subgoal_shaping_oracle
)

print(f"✓ Oracle & Verified Fix Evaluator sẵn sàng theo profile: {SCORING_PROFILE}")
print(f"  -> Probe episodes: {PROBE_EPISODES}, tail_thr: {PROBE_TAIL_THR}, gamma: {PROBE_GAMMA}, opt_init: {OPTIMISTIC_INIT}")


# CELL 4: QUỸ ĐẠO THAM CHIẾU, COUNTEREXAMPLE POOL & SIGNAL CHECKER (TỪ CEGR)


In [ ]:
# =====================================================================
# CELL 4: QUỸ ĐẠO THAM CHIẾU, COUNTEREXAMPLE POOL & SIGNAL CHECKER
# =====================================================================
from cegr.envs.doorkey import build_expert_tables, build_reference_trajs, sample_noisy, traj_return
from cegr.synthesis.ce_pool import CEPool, make_ce, Counterexample
from cegr.synthesis.signal_checker import signal_check, SignalChecker
from cegr.synthesis.dynamic_anchors import DynamicAnchorsManager
from cegr.synthesis.static_antihack import static_checks
from cegr.synthesis.runner import Kit, build_kit

# Khởi tạo Kit tham chiếu độc lập 3 luồng seed tách biệt hoàn toàn
kit = build_kit(env, ce_seed=CE_STREAM_SEED, signal_seed=SIGNAL_STREAM_SEED, eval_seed=EVAL_STREAM_SEED)
print(f"✓ Khởi tạo Kit tham chiếu hoàn tất:")
print(f"  -> {len(kit.p0)} cặp Counterexample P0 chuẩn")
print(f"  -> {len(kit.signal_trajs)} quỹ đạo Signal Stream (Seed={SIGNAL_STREAM_SEED})")
print(f"  -> {len(kit.eval_trajs)} quỹ đạo Eval Stream (Seed={EVAL_STREAM_SEED})")


# CELL 5: GATEKEEPER 5 CỔNG, ATTACKER, ASSIMILATOR & PROTOCOL RUNNERS (TỪ CEGR)


In [ ]:
# =====================================================================
# CELL 5: GATEKEEPER 5 CỔNG & CÁC RUNNER PHƯƠNG PHÁP
# =====================================================================
from cegr.synthesis.gatekeeper import GateKeeper, run_gatekeeper, build_feedback
from cegr.synthesis.assimilator import extract_experience, InvariantAssimilator
from cegr.synthesis.runner import CegrRunner
from cegr.baselines.eureka.runner import EurekaRunner
from cegr.baselines.card.runner import CardRunner
from cegr.baselines.orig_runner import OrigRunner

gk = GateKeeper(env, kit)
print("✓ GateKeeper 5 cổng, DP Attacker và Invariant Assimilator đã sẵn sàng.")
print("✓ Các Runner (CEGR, EurekaRunner, CardRunner, OrigRunner) đã được nạp sẵn sàng.")


# CELL 6: ĐỘNG CƠ vLLM BATCH INFERENCE (TỪ CEGR)


In [ ]:
# =====================================================================
# CELL 6: ĐỘNG CƠ vLLM BATCH INFERENCE
# =====================================================================
from cegr.engine.llm_vllm import init_llm, llm_batch_chat, extract_code

llm_inst = init_llm(model_id=MODEL_ID, max_model_len=MAX_MODEL_LEN)
print(f"✓ Động cơ vLLM khởi tạo thành công: {MODEL_ID} (max_model_len={MAX_MODEL_LEN})")


# CELL 7: PHA 1 (BUILD DATASET) - 12 HỌ MUTANTS & NATURAL REWARDS (TỪ CEGR)


In [ ]:
# =====================================================================
# CELL 7: PHA 1 (BUILD DATASET) - 12 HỌ MUTANTS & NATURAL REWARDS
# =====================================================================
from cegr.synthesis.dataset import (
    build_calibrated_mutants, generate_natural_rewards,
    MUTANT_FAMILIES, NATURAL_PERSONAS
)

base_manifest_path = os.path.join(OUT_DIR, "manifest.json" if ENV_NAME == "doorkey5" else f"manifest_{ENV_NAME}.json")

if os.path.exists(base_manifest_path):
    with open(base_manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    print(f"✓ Nạp manifest có sẵn: {len(manifest)} rewards từ {base_manifest_path}")
else:
    print(f"--- Đang sinh dataset cho môi trường {ENV_NAME} ---")
    mut_list = build_calibrated_mutants(env)
    print(f"  -> Đã sinh {len(mut_list)} calibrated mutants qua 12 họ.")
    
    try:
        nat_list, gen_stats = generate_natural_rewards(
            env, n_prompts=N_NATURAL_PROMPTS, seed=SEED,
            task_desc=cegr.envs.doorkey.TASK_DESC, api_desc=cegr.envs.doorkey.API_DESC
        )
        print(f"  -> Đã sinh {len(nat_list)} natural hacked rewards. Thống kê: {gen_stats}")
    except Exception as e:
        print(f"  -> Lưu ý: Không khởi tạo được vLLM batch chat ({e}), sử dụng tập starter mutants.")
        nat_list = []
    
    rng = random.Random(SEED)
    rng.shuffle(mut_list)
    rng.shuffle(nat_list)
    for idx, r in enumerate(mut_list):
        r["split"] = "pilot" if idx < PILOT_N else "main"
    for idx, r in enumerate(nat_list):
        r["split"] = "pilot" if idx < PILOT_N else "main"
        
    manifest = [r for r in mut_list + nat_list if r["split"] == "main"]
    with open(base_manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    print(f"✓ Đã lưu {len(manifest)} rewards vào {base_manifest_path}")


# CELL 8: PHA 2 - VÒNG LẶP ĐỐI SÁNH CEGR, BASELINE & ABLATION (TỪ CEGR)


In [ ]:
# =====================================================================
# CELL 8: PHA 2 - CHẠY THỰC NGHIỆM ĐỐI SÁNH CEGR, BASELINE & ABLATION
# =====================================================================
results_file = out_path("results_main", ".jsonl")
all_results = []

# Tải checkpoint kết quả đã có nếu chạy nối tiếp
if os.path.exists(results_file):
    with open(results_file, "r", encoding="utf-8") as f:
        all_results = [json.loads(line) for line in f if line.strip()]
    print(f"✓ Nạp checkpoint kết quả: {len(all_results)} jobs đã hoàn tất.")

# Cấu hình runner đồng nhất ngân sách 4 LLM calls
runner_config = {
    "max_rounds": N_ROUNDS,
    "llm": {"temperature_repair": REPAIR_TEMP, "max_tokens": REPAIR_MAXTOK},
    "scoring_profile": SCORING_PROFILE,
    "keypos_mode": KEYPOS_MODE,
}

runners = {
    "ORIG": OrigRunner(env, runner_config),
    "EUREKA": EurekaRunner(env, runner_config),
    "CARD": CardRunner(env, runner_config, use_reflection_memory=False),
    "CARD_REFL": CardRunner(env, runner_config, use_reflection_memory=True),
    "CEGR": CegrRunner(env, runner_config, method_name="CEGR"),
    "CEGR_noPool": CegrRunner(env, runner_config, method_name="CEGR_noPool", variant_cfg={"pool": False}),
    "CEGR_noAnchors": CegrRunner(env, runner_config, method_name="CEGR_noAnchors", variant_cfg={"anchors": "none"}),
    "CEGR_noClass": CegrRunner(env, runner_config, method_name="CEGR_noClass", variant_cfg={"classify": False}),
    "CEGR_noG2": CegrRunner(env, runner_config, method_name="CEGR_noG2", variant_cfg={"gates": [1, 3, 4, 5]}),
    "CEGR_noG4": CegrRunner(env, runner_config, method_name="CEGR_noG4", variant_cfg={"gates": [1, 2, 3, 5]}),
    "CEGR_noG5": CegrRunner(env, runner_config, method_name="CEGR_noG5", variant_cfg={"gates": [1, 2, 3, 4]}),
}

# 1. Chạy 4 phương pháp chính
eval_records = manifest[:N_MAIN_EVAL]
for cond in METHODS:
    already_run = [r for r in all_results if r.get("method") == cond]
    if len(already_run) >= len(eval_records) * N_SAMPLES:
        print(f"✓ Bỏ qua {cond}: đã có đủ {len(already_run)} kết quả.")
        continue
    print(f"\n>>> Đang chạy phương pháp: {cond} trên {len(eval_records)} rewards...")
    runner = runners[cond]
    cond_res = runner.run_wave_batch(eval_records, n_samples=N_SAMPLES, label=cond) if hasattr(runner, "run_wave_batch") else runner.run_batch(eval_records, n_samples=N_SAMPLES)
    all_results.extend(cond_res)
    with open(results_file, "w", encoding="utf-8") as f:
        for item in all_results:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"✓ Hoàn thành {cond}: {len(cond_res)} jobs ghi nhận.")

# 2. Chạy 6 biến thể Ablation
abl_records = manifest[:N_ABL_EVAL]
for abl in ABLATIONS:
    already_run = [r for r in all_results if r.get("method") == abl]
    if len(already_run) >= len(abl_records) * N_SAMPLES:
        print(f"✓ Bỏ qua {abl}: đã có đủ {len(already_run)} kết quả.")
        continue
    print(f"\n>>> Đang chạy biến thể: {abl} trên {len(abl_records)} rewards...")
    runner = runners[abl]
    abl_res = runner.run_wave_batch(abl_records, n_samples=N_SAMPLES, label=abl) if hasattr(runner, "run_wave_batch") else runner.run_batch(abl_records, n_samples=N_SAMPLES)
    all_results.extend(abl_res)
    with open(results_file, "w", encoding="utf-8") as f:
        for item in all_results:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"✓ Hoàn thành {abl}: {len(abl_res)} jobs ghi nhận.")

print(f"\n✓ Hoàn tất Pha 2: Tổng cộng {len(all_results)} kết quả thực nghiệm được lưu tại {results_file}.")


# CELL 9: PHA 3 - BOOTSTRAP THỐNG KÊ, BIỂU ĐỒ & BẢNG LATEX (TỪ CEGR)


In [ ]:
# =====================================================================
# CELL 9: PHA 3 - BOOTSTRAP THỐNG KÊ, BIỂU ĐỒ & XUẤT BẢNG LATEX
# =====================================================================
from cegr.eval.bootstrap import bootstrap_ci, per_reward, paired_test
from cegr.eval.report_generator import generate_latex_tables, plot_evaluation_figures

df = pd.DataFrame(all_results)
if not df.empty:
    print("Thống kê mẫu kết quả theo method:")
    cols_show = [c for c in ["vf", "tier1_safe", "loose_dp", "llm_calls", "rl_episodes"] if c in df.columns]
    print(df.groupby("method")[cols_show].mean())

    # Vẽ biểu đồ kết quả (300 DPI)
    plot_evaluation_figures(df, out_dir=OUT_DIR)
    
    # Xuất các bảng LaTeX chuẩn ICLR/IEEE
    out_tex_path = out_path("tables_report", ".tex")
    generate_latex_tables(df, out_tex_path=out_tex_path)
    print(f"✓ Đã lưu biểu đồ và các bảng LaTeX tại {OUT_DIR}.")
else:
    print("Chưa có dữ liệu kết quả để xuất biểu đồ.")


# CELL 10: TEST TOÀN VẸN VÀ NGHIỆM THU TỰ ĐỘNG


In [ ]:
# =====================================================================
# CELL 10: TEST TOÀN VẸN VÀ NGHIỆM THU TỰ ĐỘNG
# =====================================================================
print(f">>> Bắt đầu chạy test nghiệm thu trên môi trường {ENV_NAME}...")
assert env.n > 0, f"State count invalid: {env.n}"
assert kit is not None and len(kit.p0) >= 4, "Kit P0 chưa khởi tạo đúng"
assert len(runners) >= 10, "Số lượng runners không đầy đủ"

# Test thử một reward hàm cơ bản
test_code = (
    "def reward_components(s, a, s2):\n"
    "    return {'step': -0.01, 'goal': 1.0 if at_goal(s2) else 0.0}\n"
)
tab = build_reward_table(env, test_code)
Vs, Vf, _, _ = solve(env, tab, T=env.horizon)
print(f"✓ Test mẫu chuẩn DP: V_succ={Vs:.3f}, V_fail={Vf:.3f}")
print(f"✓ Toàn bộ kiểm thử nghiệm thu trên {ENV_NAME} đã hoàn thành tốt đẹp!")
